# Graded Challenge 6

**Nama:** Syifa Mutaalia  
**Batch:** 001

Program ini dibuat untuk melakukan proses Extract, Transform, dan Load (ETL) data penjualan The Look menggunakan PySpark untuk kebutuhan Data Warehouse dan analisis penjualan.



## Data Warehouse Design

Data Warehouse ini berfokus pada proses penjualan pada platform
e-commerce TheLook. Data yang digunakan berkaitan dengan transaksi
penjualan, pelanggan, produk, waktu transaksi, dan distribution center.

Proses bisnis dimulai ketika customer melakukan pembelian produk
melalui platform e-commerce. Setiap produk yang dibeli tercatat
sebagai order item dan menjadi bagian dari transaksi penjualan.

Data penjualan kemudian digunakan untuk menganalisis jumlah produk
yang terjual, total penjualan, biaya produk, profit, serta penjualan
berdasarkan customer, produk, waktu, dan distribution center.

Fact table yang digunakan adalah `fact_sales`.

Grain dari fact table adalah satu baris merepresentasikan satu
order item atau satu item produk dalam suatu order.
Measure yang disimpan:
- quantity
- sale_price
- total_sales
- product_cost
- profit

# Dimension yang digunakan:

1. `dim_customer`
   - customer_id
   - first_name
   - last_name
   - gender
   - age
   - city
   - state
   - country
   - traffic_source

2. `dim_product`
   - product_id
   - product_name
   - category
   - brand
   - department
   - sku
   - retail_price
   - product_cost
   - distribution_center_id

3. `dim_distribution_center`
   - distribution_center_id
   - distribution_center_name
   - latitude
   - longitude

4. `dim_date`
   - date_key
   - full_date
   - day
   - month
   - month_name
   - quarter
   - year

### Schema Data Warehouse

Data Warehouse menggunakan Star Schema karena terdapat satu
fact table, yaitu `fact_sales`, yang terhubung langsung dengan
dimension table.

Relasi:
- `fact_sales` → `dim_customer`
- `fact_sales` → `dim_product`
- `fact_sales` → `dim_distribution_center`
- `fact_sales` → `dim_date`

## 1. Install dan Import Package

Menyiapkan PySpark dan library yang digunakan untuk proses ETL.

In [1]:
!pip install pyspark


In [2]:
import os
from pathlib import Path

HADOOP_HOME = Path(r"D:\latihan_tugas_hacktive\hadoop")
WINUTILS = HADOOP_HOME / "bin" / "winutils.exe"

assert WINUTILS.is_file(), f"winutils.exe tidak ditemukan: {WINUTILS}"

os.environ["HADOOP_HOME"] = str(HADOOP_HOME)
os.environ["hadoop.home.dir"] = str(HADOOP_HOME)
os.environ["PATH"] = (
    str(HADOOP_HOME / "bin") + os.pathsep + os.environ["PATH"]
)

print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print("WINUTILS =", WINUTILS)

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, date_format, dayofmonth, month,
    quarter, year, lit
)

spark = (
    SparkSession.builder
    .appName("TheLookDataWarehouse")
    .master("local[2]")
    .config("spark.hadoop.io.native.lib.available", "false")
    .getOrCreate()
)

print("PySpark:", pyspark.__version__)

HADOOP_HOME = D:\latihan_tugas_hacktive\hadoop
WINUTILS = D:\latihan_tugas_hacktive\hadoop\bin\winutils.exe
PySpark: 4.2.0


## 2. Extract

Membaca file CSV hasil query dari BigQuery ke dalam PySpark DataFrame.

In [3]:
df_customer = spark.read.csv(
    "data/dim_customer.csv",
    header=True,
    inferSchema=True
)

df_product = spark.read.csv(
    "data/dim_product.csv",
    header=True,
    inferSchema=True
)

df_distribution_center = spark.read.csv(
    "data/dim_distribution_center.csv",
    header=True,
    inferSchema=True
)

df_sales = spark.read.csv(
    "data/fact_sales.csv",
    header=True,
    inferSchema=True
)

print("Customer :", df_customer.count())
print("Product :", df_product.count())
print("Distribution Center :", df_distribution_center.count())
print("Sales :", df_sales.count())

Customer : 100000
Product : 29120
Distribution Center : 10
Sales : 68236


## proses EDA

In [4]:
df_customer.printSchema()
df_customer.show()

#ceknull
from pyspark.sql.functions import sum, when
df_customer.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_customer.columns]).show()

df_product.printSchema()
df_product.show()

#ceknull
df_product.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_product.columns]).show()

df_distribution_center.printSchema()
df_distribution_center.show()

#ceknull
df_distribution_center.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_distribution_center.columns]).show()

df_sales.printSchema()
df_sales.show()

#ceknull
df_sales.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_sales.columns]).show()




root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- traffic_source: string (nullable = true)

+-----------+----------+---------+------+---+--------+-----+-------+--------------+
|customer_id|first_name|last_name|gender|age|    city|state|country|traffic_source|
+-----------+----------+---------+------+---+--------+-----+-------+--------------+
|      73789|     Henry|    Smith|     M| 45|    null| Acre| Brasil|        Search|
|      94825|     Traci|   Landry|     F| 24|    null| Acre| Brasil|        Search|
|      17823|   Bradley| Gonzales|     M| 42|    null| Acre| Brasil|        Search|
|      14473|     James|  Stevens|     M| 14|    null| Acre| Brasil|        Search|
|      46719|       Jon|    Nolan|     M| 18|  

## 3. Transform dengan PySpark

Melakukan cleaning dan transformation sesuai kebutuhan dimension dan fact table.

In [6]:
# Dimension Customer
df_customer_clean = (
    df_customer
    .dropDuplicates(["customer_id"])
    .filter(col("customer_id").isNotNull())
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
)

# Dimension Product
df_product_clean = (
    df_product
    .dropDuplicates(["product_id"])
    .filter(col("product_id").isNotNull())
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("brand", trim(col("brand")))
)

# Dimension Distribution Center
df_distribution_clean = (
    df_distribution_center
    .dropDuplicates(["dc_id"])
    .filter(col("dc_id").isNotNull())
)

# Fact Sales
from pyspark.sql.functions import col, lit, date_format

df_sales_clean = (
    df_sales
    .dropDuplicates(["order_item_id"])
    .filter(col("order_item_id").isNotNull())
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("created_at").isNotNull())
    .withColumn("sales_date", col("created_at").cast("date"))
    .withColumn("quantity", lit(1))
)


### 3.1 Membentuk kolom date_key pada Fact Sales

`date_key` digunakan sebagai foreign key ke dimension date.

In [7]:
df_sales_clean = (
    df_sales_clean
    .withColumn(
        "date_key",
        date_format(col("sales_date"), "yyyyMMdd").cast("int")
    )
)


## 4. Membuat Dimension Date

Dimension date dibentuk dari tanggal penjualan yang terdapat pada fact sales.

In [8]:
df_date = (
    df_sales_clean
    .select("sales_date")
    .dropDuplicates()
    .withColumn(
        "date_key",
        date_format(col("sales_date"), "yyyyMMdd").cast("int")
    )
    .withColumn("full_date", col("sales_date"))
    .withColumn("day", dayofmonth(col("sales_date")))
    .withColumn("month", month(col("sales_date")))
    .withColumn("month_name", date_format(col("sales_date"), "MMMM"))
    .withColumn("quarter", quarter(col("sales_date")))
    .withColumn("year", year(col("sales_date")))
    .drop("sales_date")
)
df_date.show()


+--------+----------+---+-----+----------+-------+----+
|date_key| full_date|day|month|month_name|quarter|year|
+--------+----------+---+-----+----------+-------+----+
|20210127|2021-01-27| 27|    1|   January|      1|2021|
|20220328|2022-03-28| 28|    3|     March|      1|2022|
|20260213|2026-02-13| 13|    2|  February|      1|2026|
|20250216|2025-02-16| 16|    2|  February|      1|2025|
|20251021|2025-10-21| 21|   10|   October|      4|2025|
|20260818|2026-08-18| 18|    8|    August|      3|2026|
|20230715|2023-07-15| 15|    7|      July|      3|2023|
|20240918|2024-09-18| 18|    9| September|      3|2024|
|20210622|2021-06-22| 22|    6|      June|      2|2021|
|20210827|2021-08-27| 27|    8|    August|      3|2021|
|20211113|2021-11-13| 13|   11|  November|      4|2021|
|20211218|2021-12-18| 18|   12|  December|      4|2021|
|20230622|2023-06-22| 22|    6|      June|      2|2023|
|20211011|2021-10-11| 11|   10|   October|      4|2021|
|20200824|2020-08-24| 24|    8|    August|      

## 5. Menyiapkan Data untuk Data Warehouse

Kolom disusun mengikuti struktur dimension dan fact table pada DDL.

In [9]:
df_customer_load = df_customer_clean.select(
    "customer_id",
    "first_name",
    "last_name",
    "gender",
    "age",
    "city",
    "state",
    "country",
    "traffic_source"
)

df_product_load = df_product_clean.select(
    "product_id",
    "product_name",
    "category",
    "brand",
    "department",
    "sku",
    col("cost").alias("product_cost"),
    "retail_price",
    "distribution_center_id"
)

df_distribution_load = df_distribution_clean.select(
    col("dc_id").alias("distribution_center_id"),
    col("dc_name").alias("distribution_center_name"),
    "latitude",
    "longitude"
)

df_sales_export = (
    df_sales_clean
    .join(
        df_product_clean.select(
            "product_id",
            "cost",
            "distribution_center_id"
        ),
        on="product_id",
        how="left"
    )
    .withColumn(
        "total_sales",
        col("sale_price") * col("quantity")
    )
    .withColumn(
        "profit",
        (col("sale_price") - col("cost")) * col("quantity")
    )
    .select(
        "order_item_id",
        "order_id",
        "customer_id",
        "product_id",
        "distribution_center_id",
        "date_key",
        "quantity",
        "sale_price",
        "total_sales",
        col("cost").alias("product_cost"),
        "profit"
    )
)

df_sales_export.show()
df_sales_export.printSchema()


+-------------+--------+-----------+----------+----------------------+--------+--------+-----------------+-----------------+------------------+------------------+
|order_item_id|order_id|customer_id|product_id|distribution_center_id|date_key|quantity|       sale_price|      total_sales|      product_cost|            profit|
+-------------+--------+-----------+----------+----------------------+--------+--------+-----------------+-----------------+------------------+------------------+
|       136625|   94200|      76058|     28551|                     5|20240801|       1|2.990000009536743|2.990000009536743|1.1870299992526694|1.8029700102840738|
|        38311|   26435|      21254|     14274|                     3|20220223|       1|3.990000009536743|3.990000009536743|1.5720599976038188|2.4179400119329246|
|        15790|   10842|       8769|     13940|                     8|20250124|       1|4.150000095367432|4.150000095367432|1.7513000335663556| 2.398700061801076|
|       124861|   8610

## 6. Export Hasil Transformasi

Data hasil transformasi PySpark disimpan sebagai CSV untuk proses load ke PostgreSQL menggunakan SQL DML.

In [10]:
import os
import csv

def export_csv(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(df.columns)

        for row in df.toLocalIterator():
            writer.writerow(row)

    print("Berhasil:", path)


export_csv(df_customer_load, "output/dim_customer/dim_customer.csv")
export_csv(df_product_load, "output/dim_product/dim_product.csv")
export_csv(df_distribution_load, "output/dim_distribution_center/dim_distribution_center.csv")
export_csv(df_date, "output/dim_date/dim_date.csv")
export_csv(df_sales_export, "output/fact_sales/fact_sales.csv")

print("EXPORT SELESAI")

Berhasil: output/dim_customer/dim_customer.csv
Berhasil: output/dim_product/dim_product.csv
Berhasil: output/dim_distribution_center/dim_distribution_center.csv
Berhasil: output/dim_date/dim_date.csv
Berhasil: output/fact_sales/fact_sales.csv
EXPORT SELESAI


## 7. Validasi Hasil Transformasi

Mengecek jumlah baris hasil transformasi sebelum data dimuat ke PostgreSQL.


In [11]:
print("Customer :", df_customer_load.count())
print("Product :", df_product_load.count())
print("Distribution Center :", df_distribution_load.count())
print("Date :", df_date.count())
print("Fact Sales :", df_sales_export.count())

print("\nSchema Fact Sales:")
df_sales_export.printSchema()


Customer : 100000
Product : 29120
Distribution Center : 10
Date : 2694
Fact Sales : 68236

Schema Fact Sales:
root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- distribution_center_id: integer (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- quantity: integer (nullable = false)
 |-- sale_price: double (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- product_cost: double (nullable = true)
 |-- profit: double (nullable = true)



## 8. Load ke PostgreSQL

Tahap load dilakukan menggunakan SQL DML `COPY` pada file `thelook_dwh_dml_syifa.sql`.

Urutan load:
1. `dim_customer`
2. `dim_product`
3. `dim_distribution_center`
4. `dim_date`
5. `fact_sales`



## 9. kode untuk dbdiagram.io

In [ ]:
#bentuk dbdiagram
Table dim_customer {
  customer_key int [pk, increment]
  customer_id int [unique, not null]
  first_name varchar
  last_name varchar
  gender varchar
  age int
  city varchar
  state varchar
  country varchar
  traffic_source varchar
}

Table dim_product {
  product_key int [pk, increment]
  product_id int [unique, not null]
  product_name varchar
  category varchar
  brand varchar
  department varchar
  sku varchar
  retail_price decimal
  product_cost decimal
  distribution_center_id int
}

Table dim_distribution_center {
  distribution_center_key int [pk, increment]
  distribution_center_id int [unique, not null]
  distribution_center_name varchar
  latitude decimal
  longitude decimal
}

Table dim_date {
  date_key int [pk]
  full_date date
  day int
  month int
  month_name varchar
  quarter int
  year int
}

Table fact_sales {
  sales_key bigint [pk, increment]
  order_item_id int
  order_id int
  customer_key int
  product_key int
  distribution_center_key int
  date_key int
  quantity int
  sale_price decimal
  total_sales decimal
  product_cost decimal
  profit decimal
}

Ref: fact_sales.customer_key > dim_customer.customer_key
Ref: fact_sales.product_key > dim_product.product_key
Ref: fact_sales.distribution_center_key > dim_distribution_center.distribution_center_key
Ref: fact_sales.date_key > dim_date.date_key